# Desarrollo de Entrega Unidad 4 – Base de Datos de tienda usando las Unidades 9 y 10

En este caso trabajaremos con una base de datos de órdenes de bases de datos de tienda de la unidad 4.

Iniciemos importando las librerías necesarias

In [11]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet # Se debe instalar previamente la librería prophet
from prophet.diagnostics import cross_validation, performance_metrics
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# Cargando y Analizando la Base de datos
Ahora importamos nuestra base de datos

In [12]:
consumer_data = pd.read_excel('data/Online Retail.xlsx')
consumer_data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


# Cambiar los nombres de la base de datos creando un directorio 

In [13]:
from os import rename
consumer_data.rename(columns={'InvoiceNo':'Numero_factura','StockCode':'Cod_prod','Description':'Nombre_Producto','Quantity':'Cantidad','UnitPrice':'Precio unitario','InvoiceDate':'Fecha_compra','CustomerID':'Cod_Cliente','Country':'País'},inplace=True)
consumer_data = pd.DataFrame(consumer_dat["Numero_factura"].astype(str))
consumer_data.to_csv('data/Online_Retail.csv')
consumer_data

NameError: name 'consumer_dat' is not defined

Se analiza la estadistica de las variables númericas

In [ ]:
consumer_data.describe()

,Numero_factura
count,541909
unique,25900
top,573585
freq,1114


Tipo de datos que se tienen

In [ ]:
consumer_data.dtypes

Numero_factura             object
Cod_prod                   object
Nombre_Producto            object
Cantidad                    int64
Fecha_compra       datetime64[us]
Precio unitario           float64
Cod_Cliente               float64
País                          str
dtype: object

In [ ]:
print('Cantidad de paises: ', consumer_data['País'].nunique())
print('Cantidad de productos: ', consumer_data['Cod_prod'].nunique())
print('Cantidad de usuarios: ', consumer_data['Cod_Cliente'].nunique())

Cantidad de paises:  38
Cantidad de productos:  4070
Cantidad de usuarios:  4372


Verifiquemos si tenemos valores faltantes en nuestra base de datos

In [ ]:
consumer_data.isna().sum()

Numero_factura          0
Cod_prod                0
Nombre_Producto      1454
Cantidad                0
Fecha_compra            0
Precio unitario         0
Cod_Cliente        135080
País                    0
dtype: int64

# Tratamiento de la base de Datos
Teniendo en cuenta el código del producto se rellenaran los espacios

In [ ]:
def completar_nombres_productos(df):
    """
    Rellena los espacios en blanco (valores nulos) de la columna 'Nombre_Producto' 
    buscando el nombre asociado a su 'Cod_prod' en el resto del DataFrame.
    """
    # 1. Creamos un mapeo (diccionario) de Cod_prod -> Nombre_Producto
    # Descartamos los nulos y nos quedamos con la primera aparición de cada producto
    mapeo_nombres = (
        df[['Cod_prod', 'Nombre_Producto']]
        .dropna(subset=['Nombre_Producto'])
        .drop_duplicates(subset=['Cod_prod'])
        .set_index('Cod_prod')['Nombre_Producto']
    )
    
    # 2. Rellenamos los valores nulos en 'Nombre_Producto' usando la función map() 
    # que busca el Cod_prod en nuestro mapeo
    df['Nombre_Producto'] = df['Nombre_Producto'].fillna(df['Cod_prod'].map(mapeo_nombres))
    
    return df

# funcion adicional 


# Ejemplo de uso con tu DataFrame:
consumer_data = completar_nombres_productos(consumer_data)

# Para verificar que funcionó (opcional):
print("Valores nulos en 'Nombre_Producto':", consumer_data['Nombre_Producto'].isnull().sum())


Valores nulos en 'Nombre_Producto': 112


Verificamos el resultado

In [ ]:
consumer_data.isna().sum()

Numero_factura          0
Cod_prod                0
Nombre_Producto       112
Cantidad                0
Fecha_compra            0
Precio unitario         0
Cod_Cliente        135080
País                    0
dtype: int64

In [ ]:
consumer_data

,Numero_factura,Cod_prod,Nombre_Producto,Cantidad,Fecha_compra,Precio unitario,Cod_Cliente,País
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


Anexando una columna de estado de la compra

In [15]:
def asignar_estado_compra(df):
    """
    Crea una nueva columna 'Estado de la compra'.
    Si el 'Numero_factura' contiene la letra 'C', se marca como 'Cancelado', 
    de lo contrario se marca como 'Realizado'.
    """
    # 1. Nos aseguramos de tratar la columna Numero_factura como texto (string)
    # 2. Comprobamos si contiene la letra 'C' (case=False para que no distinga entre mayúsculas y minúsculas)
    condicion_cancelado = df['Numero_factura'].astype(str).str.contains('C', case=False, na=False)
    
    # 3. np.where funciona así: np.where(condicion, valor_si_verdadero, valor_si_falso)
    df['Estado de la compra'] = np.where(condicion_cancelado, 'Cancelado', 'Realizado')
    
    return df
# Ejemplo de uso con tu DataFrame:
consumer_data = asignar_estado_compra(consumer_data)
# Para verificar que funcionó viendo algunas filas canceladas:
display(consumer_data[consumer_data['Estado de la compra'] == 'Cancelado'].head())

,Numero_factura,Cod_prod,Nombre_Producto,Cantidad,Fecha_compra,Precio unitario,Cod_Cliente,País,Estado de la compra
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom,Cancelado
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom,Cancelado
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,Cancelado
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,Cancelado
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,Cancelado


Se adiciona la linea para trasformar la columna Numero de Factura a tipo texto.  

In [16]:
consumer_data["Numero_factura"] = consumer_data["Numero_factura"].astype(str)
consumer_data

,Numero_factura,Cod_prod,Nombre_Producto,Cantidad,Fecha_compra,Precio unitario,Cod_Cliente,País,Estado de la compra
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,Realizado
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Realizado
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,Realizado
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Realizado
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Realizado
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,Realizado
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,Realizado
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,Realizado
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,Realizado
